# NB04 — BTCS Multi-Dataset Evaluation
**Andre da Costa Silva | ITA | 2026**

Pipeline genérica para avaliar BTCS em múltiplos datasets:

| Dataset | Tipo label | Edges | Temporal | GNN? |
|---------|-----------|-------|----------|------|
| AML100k | edge (GNN) | 2.49M | sim | pré-treinado |
| AML1M | edge (GNN) | ~24M | sim | pré-treinado |
| Bitcoin OTC | edge (ratings) | 35k | sim | não |
| Bitcoin Alpha | edge (ratings) | 24k | sim | não |
| IBM AML txns | edge (IS_FRAUD) | 1.3M | sim | não |
| IBM HI-Small | edge (Is Laundering) | 5M | sim | não |
| Elliptic | node → edge | 234k | sim | treinar |
| PaySim | txn-level | ~6M | sim | treinar |
| Amazon Fraud | node → edge (.mat) | 8.8M | não | treinar |
| Yelp Fraud | node → edge (.mat) | 7.7M | não | treinar |

Executar: células 1 → ... → N (ou Runtime → Run all)

**Pré-requisito:** rodar `scripts/download_datasets.py` ou baixar manualmente antes.

In [ ]:
# CELULA 1 - Setup
import os, sys, subprocess, time, math, contextlib, warnings
from pathlib import Path
from collections import defaultdict
warnings.filterwarnings('ignore')

def _pip(*pkgs):
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q'] + list(pkgs))

_pip('numpy', 'pandas', 'scikit-learn', 'matplotlib', 'psutil',
     'python-igraph', 'leidenalg', 'scipy')
try:
    import torch
except ImportError:
    _pip('torch', '--index-url', 'https://download.pytorch.org/whl/cu121')
    import torch

import numpy as np
import pandas as pd
import psutil
import igraph as ig
import leidenalg
import matplotlib.pyplot as plt
from IPython.display import display

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)

BASE = Path('/content/drive/MyDrive') if IN_COLAB else Path('.').resolve()
DATA = BASE / 'GrafosGNN/data'
OUT  = BASE / 'GrafosGNN/results/nb04_multi_dataset'
OUT.mkdir(parents=True, exist_ok=True)

# Paths AML datasets (pre-treinados)
AML100K_BASE  = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML100k'
AML1M_BASE    = BASE / 'DatasetDissertacao/IBM_TRANSACTION_AML/AMLSIMFULL/AML1M'

print(f'Base: {BASE}')
print(f'Data: {DATA}')
print(f'Out:  {OUT}')

# Verificar datasets disponíveis
print('\n=== Datasets disponíveis ===')
for name in ['bitcoin_otc', 'bitcoin_alpha', 'paysim', 'elliptic',
             'ibm_aml', 'amazon_fraud', 'yelp_fraud', 'credit_card_fraud',
             'dgraph_fin', 'ethereum_phishing', 't_finance']:
    d = DATA / name
    if not d.exists():
        print(f"  {name:25s} NAO EXISTE")
        continue
    files = [f for f in d.rglob('*') if f.is_file() and f.name != '.gitkeep']
    total_mb = sum(f.stat().st_size for f in files) / 1e6
    status = f'{len(files)} files ({total_mb:,.0f} MB)' if files else 'VAZIO'
    print(f"  {name:25s} {status}")

In [ ]:
# CELULA 2 - Funcoes utilitarias

@contextlib.contextmanager
def measure_resources():
    proc = psutil.Process(os.getpid())
    m0 = proc.memory_info().rss / 1024**2
    t0 = time.perf_counter()
    r  = {}
    yield r
    r['time_s'] = time.perf_counter() - t0
    r['ram_mb']  = proc.memory_info().rss / 1024**2 - m0


def evaluate_cases_generic(cases, y_edges, n_test, k_frac):
    """Avalia casos para qualquer dataset.
    
    Args:
        cases: lista de dicts com 'nodes', 'seed_edges', 'induced_edges'
        y_edges: array de labels (0/1) para TODAS as arestas usadas como 'full'
        n_test: numero total de arestas de teste
        k_frac: fracao k usada
    """
    y = np.asarray(y_edges, dtype=int)
    pos_total = int(y.sum())
    K = max(1, int(math.ceil(k_frac * n_test)))
    
    if not cases:
        return {m: np.nan for m in [
            'n_cases','coverage','purity_induced','ocr_b100',
            'edges_per_case_median','edges_per_case_max','e_ind_total']}
    
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    non_empty = [c['induced_edges'] for c in cases if c['induced_edges']]
    if non_empty:
        all_ind = np.unique(np.concatenate(
            [np.asarray(x, dtype=np.int64) for x in non_empty]))
    else:
        all_ind = np.array([], dtype=np.int64)
    
    # Clamp indices
    valid = all_ind[all_ind < len(y)]
    pos_ind = int(y[valid].sum()) if len(valid) > 0 else 0
    coverage = float(pos_ind / max(pos_total, 1))
    purity = float(pos_ind / max(int(ind_sizes.sum()), 1))
    
    return {
        'n_cases': len(cases),
        'coverage': coverage,
        'purity_induced': purity,
        'ocr_b100': float((ind_sizes > 100).mean()),
        'edges_per_case_median': float(np.median(ind_sizes)),
        'edges_per_case_max': float(ind_sizes.max()),
        'e_ind_total': int(ind_sizes.sum()),
    }

print('Funcoes utilitarias definidas.')

In [ ]:
# CELULA 3 - BTCS v3 generico
# Aceita qualquer dataset via interface padronizada:
#   scores: array de probabilidades/scores por aresta
#   src, dst: arrays de endpoints
#   timestamps: array de timestamps (int64)
#   y: labels ground truth (0/1)

def build_Lk(src_sel, dst_sel, ts_sel, delta_L, max_hub_edges=500):
    K = len(src_sel)
    node_map = defaultdict(list)
    for i in range(K):
        node_map[int(src_sel[i])].append((i, int(ts_sel[i])))
        node_map[int(dst_sel[i])].append((i, int(ts_sel[i])))
    lk_edges = set()
    for node, elist in node_map.items():
        if len(elist) < 2:
            continue
        if len(elist) > max_hub_edges:
            elist.sort(key=lambda x: x[1])
            elist = elist[:max_hub_edges]
        else:
            elist.sort(key=lambda x: x[1])
        for i in range(len(elist)):
            ei, ti = elist[i]
            for j in range(i+1, len(elist)):
                ej, tj = elist[j]
                if tj - ti > delta_L:
                    break
                if ei != ej:
                    lk_edges.add((min(ei, ej), max(ei, ej)))
    return list(lk_edges)


def btcs_generic(scores, src, dst, timestamps, y,
                 k=0.01, delta_L=7, resolution=1.0, budget_B=100, seed=42):
    """BTCS v3 generico: funciona com qualquer dataset.
    
    Args:
        scores: array (N,) de scores por aresta (maior = mais suspeito)
        src: array (N,) de source node IDs
        dst: array (N,) de dest node IDs
        timestamps: array (N,) de timestamps (int)
        y: array (N,) de labels ground truth
        k: fracao de top edges
        delta_L: janela temporal para Lk
        resolution: gamma do Leiden
        budget_B: max edges induzidas por caso
        seed: random seed
    
    Returns:
        cases: lista de dicts
    """
    scores = np.asarray(scores, dtype=float)
    src = np.asarray(src, dtype=np.int64)
    dst = np.asarray(dst, dtype=np.int64)
    ts = np.asarray(timestamps, dtype=np.int64)
    N = len(scores)
    K = max(1, int(math.ceil(k * N)))
    sel = np.argsort(-scores)[:K]
    
    src_sel, dst_sel, ts_sel = src[sel], dst[sel], ts[sel]
    max_node = int(max(src.max(), dst.max())) + 1
    print(f'  K={K:,} top edges (of {N:,})')
    
    # Step 1: WCC no subgrafo de nos top-k
    all_nodes = np.unique(np.concatenate([src_sel, dst_sel]))
    nmap = {int(n): i for i, n in enumerate(all_nodes)}
    edges_compact = [(nmap[int(s)], nmap[int(d)])
                     for s, d in zip(src_sel, dst_sel)]
    g_node = ig.Graph(n=len(all_nodes), edges=edges_compact, directed=False)
    g_node.simplify()
    wcc = g_node.connected_components(mode='weak')
    wcc_mem = np.array(wcc.membership, dtype=np.int64)
    n_wcc = int(wcc_mem.max()) + 1
    edge_wcc = np.array([wcc_mem[nmap[int(s)]] for s in src_sel], dtype=np.int64)
    
    # Agrupar por WCC
    wcc_edge_lists = [[] for _ in range(n_wcc)]
    wcc_node_sets = [set() for _ in range(n_wcc)]
    for i in range(K):
        w = int(edge_wcc[i])
        wcc_edge_lists[w].append(i)
        wcc_node_sets[w].update([int(src_sel[i]), int(dst_sel[i])])
    
    # Contar induzidas por WCC (vetorizado)
    wcc_gid = -np.ones(max_node, dtype=np.int64)
    for w, nodes in enumerate(wcc_node_sets):
        for nid in nodes:
            wcc_gid[nid] = w
    g_src_w = np.where(src < max_node, wcc_gid[src], -1)
    g_dst_w = np.where(dst < max_node, wcc_gid[dst], -1)
    mask_w = (g_src_w == g_dst_w) & (g_src_w >= 0)
    idx_w = np.where(mask_w)[0]
    wcc_ind_count = np.zeros(n_wcc, dtype=np.int64)
    if len(idx_w) > 0:
        gw = g_src_w[idx_w]
        uq, cnt = np.unique(gw, return_counts=True)
        wcc_ind_count[uq] = cnt
    
    n_small = int((wcc_ind_count[wcc_ind_count > 0] <= (budget_B or 1e18)).sum())
    n_large = int((wcc_ind_count > (budget_B or 1e18)).sum())
    print(f'  WCC: {n_wcc:,} comps | {n_small} small, {n_large} large')
    
    # Step 2: Hierarquico - split grandes com Leiden
    final_mem = np.full(K, -1, dtype=np.int64)
    next_id = 0
    for w in range(n_wcc):
        comp = wcc_edge_lists[w]
        if not comp:
            continue
        need_split = (budget_B is not None and wcc_ind_count[w] > budget_B
                      and len(comp) >= 2)
        if not need_split:
            for i in comp:
                final_mem[i] = next_id
            next_id += 1
        else:
            comp_arr = np.array(comp, dtype=np.int64)
            lk_local = build_Lk(src_sel[comp_arr], dst_sel[comp_arr],
                                ts_sel[comp_arr], delta_L)
            if not lk_local:
                for i in comp:
                    final_mem[i] = next_id
                    next_id += 1
            else:
                g_local = ig.Graph(n=len(comp), edges=lk_local, directed=False)
                g_local.simplify()
                part = leidenalg.find_partition(
                    g_local, leidenalg.RBConfigurationVertexPartition,
                    resolution_parameter=resolution, seed=seed)
                local_mem = np.array(part.membership, dtype=np.int64)
                n_sub = int(local_mem.max()) + 1
                for j, idx in enumerate(comp):
                    final_mem[idx] = next_id + int(local_mem[j])
                next_id += n_sub
    
    # Step 3: Montar casos (voto majoritario)
    n_total = next_id
    node_votes = defaultdict(lambda: defaultdict(int))
    for i in range(K):
        g = int(final_mem[i])
        if g < 0: continue
        node_votes[int(src_sel[i])][g] += 1
        node_votes[int(dst_sel[i])][g] += 1
    
    cases = [{'nodes': set(), 'seed_edges': [], 'induced_edges': []}
             for _ in range(n_total)]
    for nid, votes in node_votes.items():
        best = max(votes, key=votes.get)
        cases[best]['nodes'].add(nid)
    for i in range(K):
        g = int(final_mem[i])
        if g >= 0:
            cases[g]['seed_edges'].append(int(sel[i]))
    cases = [c for c in cases if c['nodes']]
    
    # Step 4: Arestas induzidas (vetorizado)
    gid_of = -np.ones(max_node, dtype=np.int64)
    for g, case in enumerate(cases):
        for nid in case['nodes']:
            gid_of[nid] = g
    g_src_f = np.where(src < max_node, gid_of[src], -1)
    g_dst_f = np.where(dst < max_node, gid_of[dst], -1)
    mask_f = (g_src_f == g_dst_f) & (g_src_f >= 0)
    idx_f = np.where(mask_f)[0]
    if len(idx_f) > 0:
        gf = g_src_f[idx_f]
        order = np.argsort(gf, kind='stable')
        g_sorted = gf[order]
        i_sorted = idx_f[order]
        uq, cnt = np.unique(g_sorted, return_counts=True)
        for g_id, grp in zip(uq, np.split(i_sorted, np.cumsum(cnt)[:-1])):
            cases[g_id]['induced_edges'] = grp.tolist()
    
    # Step 5: Budget cap
    n_capped = 0
    if budget_B is not None:
        for case in cases:
            if len(case['induced_edges']) > budget_B:
                n_capped += 1
                idx_arr = np.array(case['induced_edges'], dtype=np.int64)
                valid_idx = idx_arr[idx_arr < len(scores)]
                sc_ind = scores[valid_idx]
                top_b = valid_idx[np.argsort(-sc_ind)[:budget_B]]
                case['induced_edges'] = top_b.tolist()
    
    ind_sizes = np.array([len(c['induced_edges']) for c in cases])
    print(f'  Final: {len(cases)} cases | capped={n_capped} | '
          f'median={np.median(ind_sizes):.0f} max={ind_sizes.max()}')
    return cases

print('BTCS v3 generico definido.')

In [ ]:
# CELULA 4 - Loaders por dataset

def load_bitcoin_otc_alpha(name='bitcoin_otc'):
    """Carrega Bitcoin OTC ou Alpha do SNAP.
    Formato: SOURCE  TARGET  RATING  TIME
    Rating: -10 a +10. Negativo = suspeito.
    """
    path = DATA / name
    csvs = list(path.glob('*.csv'))
    assert csvs, f'Nenhum CSV em {path}. Rode download_datasets.py primeiro.'
    df = pd.read_csv(csvs[0], header=None,
                     names=['src', 'dst', 'rating', 'timestamp'])
    print(f'[{name}] {len(df):,} edges | '
          f'nodes={df[["src","dst"]].max().max()+1} | '
          f'neg_ratings={int((df.rating < 0).sum())} ({100*(df.rating<0).mean():.1f}%)')
    
    all_ids = pd.unique(pd.concat([df['src'], df['dst']]))
    id_map = {v: i for i, v in enumerate(sorted(all_ids))}
    src = df['src'].map(id_map).values.astype(np.int64)
    dst = df['dst'].map(id_map).values.astype(np.int64)
    
    ratings = df['rating'].values.astype(float)
    scores = -ratings
    scores = (scores - scores.min()) / (scores.max() - scores.min() + 1e-8)
    y = (ratings < 0).astype(int)
    
    timestamps = df['timestamp'].values.astype(np.int64)
    ts_days = (timestamps - timestamps.min()) // 86400
    
    print(f'  Scores: [{scores.min():.3f}, {scores.max():.3f}] | '
          f'y_pos={y.sum()} ({100*y.mean():.1f}%) | '
          f'ts_range={ts_days.max()} days')
    
    return {
        'name': name,
        'scores': scores, 'src': src, 'dst': dst,
        'timestamps': ts_days, 'y': y,
        'n_nodes': len(id_map), 'n_edges': len(df),
        'delta_L': 30,
    }


def load_paysim():
    """Carrega PaySim (Kaggle).
    Cada transacao = aresta. Score heuristico (type_risk * amount_norm).
    """
    path = DATA / 'paysim'
    csvs = list(path.glob('*.csv'))
    assert csvs, f'Nenhum CSV em {path}. Rode download_datasets.py primeiro.'
    df = pd.read_csv(csvs[0])
    print(f'[PaySim] {len(df):,} transacoes | cols={list(df.columns)}')
    
    all_ids = pd.unique(pd.concat([df['nameOrig'], df['nameDest']]))
    id_map = {v: i for i, v in enumerate(all_ids)}
    src = df['nameOrig'].map(id_map).values.astype(np.int64)
    dst = df['nameDest'].map(id_map).values.astype(np.int64)
    
    y = df['isFraud'].values.astype(int)
    timestamps = df['step'].values.astype(np.int64)
    
    type_risk = df['type'].map({
        'TRANSFER': 0.8, 'CASH_OUT': 0.9,
        'PAYMENT': 0.3, 'CASH_IN': 0.1, 'DEBIT': 0.2
    }).fillna(0.5).values
    amount_norm = (df['amount'].values - df['amount'].min()) / \
                  (df['amount'].max() - df['amount'].min() + 1e-8)
    scores = type_risk * amount_norm
    
    print(f'  Nodes={len(id_map):,} | Fraud={y.sum():,} ({100*y.mean():.2f}%) | '
          f'Steps={timestamps.max()}')
    print(f'  [NOTA] Scores heuristicos. Para paper, treinar GNN.')
    
    return {
        'name': 'paysim',
        'scores': scores, 'src': src, 'dst': dst,
        'timestamps': timestamps, 'y': y,
        'n_nodes': len(id_map), 'n_edges': len(df),
        'delta_L': 24,
    }


def load_elliptic():
    """Carrega Elliptic (Kaggle).
    Node-level labels -> derivar edge labels.
    Busca recursiva em subpastas (elliptic_bitcoin_dataset/).
    """
    path = DATA / 'elliptic'
    feat_csv = list(path.rglob('*features*'))
    edge_csv = list(path.rglob('*edgelist*'))
    class_csv = list(path.rglob('*classes*'))
    assert feat_csv and edge_csv and class_csv, \
        f'Arquivos Elliptic nao encontrados em {path} (nem subpastas)'
    
    df_feat = pd.read_csv(feat_csv[0], header=None)
    df_edges = pd.read_csv(edge_csv[0])
    df_class = pd.read_csv(class_csv[0])
    
    df_class.columns = ['txId', 'class']
    node_label = {}
    for _, row in df_class.iterrows():
        if row['class'] in ['1', '2', 1, 2]:
            node_label[int(row['txId'])] = 1 if str(row['class']) == '1' else 0
    
    node_ts = dict(zip(df_feat[0].astype(int), df_feat[1].astype(int)))
    
    df_edges.columns = ['src', 'dst']
    all_ids = pd.unique(pd.concat([df_edges['src'], df_edges['dst']]))
    id_map = {int(v): i for i, v in enumerate(sorted(all_ids))}
    
    src = df_edges['src'].map(id_map).values.astype(np.int64)
    dst = df_edges['dst'].map(id_map).values.astype(np.int64)
    
    y_src = np.array([node_label.get(int(s), 0) for s in df_edges['src']])
    y_dst = np.array([node_label.get(int(d), 0) for d in df_edges['dst']])
    y = np.maximum(y_src, y_dst)
    
    ts_src = np.array([node_ts.get(int(s), 0) for s in df_edges['src']])
    ts_dst = np.array([node_ts.get(int(d), 0) for d in df_edges['dst']])
    timestamps = np.maximum(ts_src, ts_dst).astype(np.int64)
    
    def node_score(nid):
        if nid in node_label:
            return 1.0 if node_label[nid] == 1 else 0.0
        return 0.5
    s_src = np.array([node_score(int(s)) for s in df_edges['src']])
    s_dst = np.array([node_score(int(d)) for d in df_edges['dst']])
    scores = np.maximum(s_src, s_dst)
    scores += np.random.RandomState(42).uniform(0, 0.01, len(scores))
    
    n_labeled = sum(1 for v in node_label.values())
    n_illicit = sum(1 for v in node_label.values() if v == 1)
    print(f'[Elliptic] {len(df_edges):,} edges | {len(id_map):,} nodes')
    print(f'  Node labels: {n_labeled} labeled ({n_illicit} illicit)')
    print(f'  Edge labels (OR): {y.sum():,} suspeitas ({100*y.mean():.1f}%)')
    print(f'  [NOTA] Scores derivados de node labels. Para paper, treinar GNN.')
    
    return {
        'name': 'elliptic',
        'scores': scores, 'src': src, 'dst': dst,
        'timestamps': timestamps, 'y': y,
        'n_nodes': len(id_map), 'n_edges': len(df_edges),
        'delta_L': 2,
    }


def load_ibm_aml_transactions():
    """Carrega IBM AML transactions.csv (AMLSim-gerado).
    ~1.3M transacoes com IS_FRAUD edge-level label.
    Colunas: TX_ID,SENDER_ACCOUNT_ID,RECEIVER_ACCOUNT_ID,TX_TYPE,TX_AMOUNT,TIMESTAMP,IS_FRAUD,ALERT_ID
    """
    csv_path = DATA / 'ibm_aml' / 'transactions.csv'
    assert csv_path.exists(), f'{csv_path} nao encontrado'
    
    df = pd.read_csv(csv_path)
    print(f'[IBM AML txns] {len(df):,} transacoes | cols={list(df.columns)}')
    
    all_ids = pd.unique(pd.concat([
        df['SENDER_ACCOUNT_ID'].astype(str),
        df['RECEIVER_ACCOUNT_ID'].astype(str)
    ]))
    id_map = {v: i for i, v in enumerate(sorted(all_ids))}
    src = df['SENDER_ACCOUNT_ID'].astype(str).map(id_map).values.astype(np.int64)
    dst = df['RECEIVER_ACCOUNT_ID'].astype(str).map(id_map).values.astype(np.int64)
    
    y = df['IS_FRAUD'].values.astype(int)
    
    ts_raw = pd.to_numeric(df['TIMESTAMP'], errors='coerce').fillna(0)
    timestamps = ts_raw.values.astype(np.int64)
    if timestamps.max() > 1e9:
        timestamps = (timestamps - timestamps.min()) // 86400
    
    amount = df['TX_AMOUNT'].values.astype(float)
    scores = (amount - amount.min()) / (amount.max() - amount.min() + 1e-8)
    scores += np.random.RandomState(42).uniform(0, 0.01, len(scores))
    
    print(f'  Nodes={len(id_map):,} | Fraud={y.sum():,} ({100*y.mean():.2f}%)')
    print(f'  Timestamps: {timestamps.min()}..{timestamps.max()}')
    
    return {
        'name': 'ibm_aml_txns',
        'scores': scores, 'src': src, 'dst': dst,
        'timestamps': timestamps, 'y': y,
        'n_nodes': len(id_map), 'n_edges': len(df),
        'delta_L': 7,
    }


def load_ibm_hi_small():
    """Carrega IBM HI-Small_Trans.csv (~5M transacoes).
    Colunas duplicadas: pandas auto-renomeia Account -> Account, Account.1
    """
    csv_path = DATA / 'ibm_aml' / 'HI-Small_Trans.csv'
    assert csv_path.exists(), f'{csv_path} nao encontrado'
    
    # Pandas auto-renomeia colunas duplicadas: Account -> Account, Account.1
    df = pd.read_csv(csv_path, header=0)
    
    print(f'[IBM HI-Small] {len(df):,} transacoes | cols={list(df.columns)}')
    
    # Criar ID unico: "Bank_Account"
    # Account = sender account, Account.1 = receiver account
    src_id = df['From Bank'].astype(str) + '_' + df['Account'].astype(str)
    dst_id = df['To Bank'].astype(str) + '_' + df['Account.1'].astype(str)
    
    all_ids = pd.unique(pd.concat([src_id, dst_id]))
    id_map = {v: i for i, v in enumerate(all_ids)}
    src = src_id.map(id_map).values.astype(np.int64)
    dst = dst_id.map(id_map).values.astype(np.int64)
    
    y = df['Is Laundering'].values.astype(int)
    
    # Timestamp: converter string para inteiro ordinal
    ts_str = df['Timestamp'].astype(str)
    ts_sorted = ts_str.sort_values().unique()
    ts_map = {v: i for i, v in enumerate(ts_sorted)}
    timestamps = ts_str.map(ts_map).values.astype(np.int64)
    
    # Score: amount normalizado
    amount = df['Amount Paid'].fillna(df['Amount Received']).values.astype(float)
    scores = (amount - amount.min()) / (amount.max() - amount.min() + 1e-8)
    scores += np.random.RandomState(42).uniform(0, 0.001, len(scores))
    
    print(f'  Nodes={len(id_map):,} | Laundering={y.sum():,} ({100*y.mean():.2f}%)')
    print(f'  [NOTA] Dataset grande (5M). Usar k pequeno se RAM limitada.')
    
    return {
        'name': 'ibm_hi_small',
        'scores': scores, 'src': src, 'dst': dst,
        'timestamps': timestamps, 'y': y,
        'n_nodes': len(id_map), 'n_edges': len(df),
        'delta_L': 30,
    }


def load_mat_fraud(name, mat_filename, adj_key='homo'):
    """Loader generico para datasets .mat (Amazon Fraud, Yelp Fraud).
    
    Formato .mat: 'homo' (adjacency sparse), 'features' (N x D), 'label' (1 x N).
    Node-level labels -> edge labels via OR.
    SEM timestamps (atribuir sequencial por indice).
    """
    import scipy.io as sio
    import scipy.sparse as sp
    
    mat_path = DATA / name / mat_filename
    assert mat_path.exists(), f'{mat_path} nao encontrado'
    
    print(f'[{name}] Carregando {mat_filename}...')
    mat = sio.loadmat(str(mat_path))
    
    adj = mat[adj_key]
    if sp.issparse(adj):
        adj = adj.tocoo()
    else:
        adj = sp.coo_matrix(adj)
    
    labels = np.asarray(mat['label']).flatten()
    n_nodes = len(labels)
    fraud_rate = labels.sum() / len(labels) * 100
    
    print(f'  {n_nodes:,} nodes | {adj.nnz:,} edges (sparse) | '
          f'fraud={labels.sum():,} ({fraud_rate:.1f}%)')
    
    # Extrair edges de upper triangle (grafo nao-direcionado)
    row, col = adj.row, adj.col
    mask_upper = row < col
    src = row[mask_upper].astype(np.int64)
    dst = col[mask_upper].astype(np.int64)
    
    n_edges = len(src)
    print(f'  Upper triangle: {n_edges:,} edges')
    
    # Edge labels: OR (edge suspeita se >= 1 endpoint e fraud)
    y = np.maximum(labels[src], labels[dst]).astype(int)
    
    # Scores: node fraud probability como proxy
    scores = np.maximum(labels[src].astype(float), labels[dst].astype(float))
    scores += np.random.RandomState(42).uniform(0, 0.01, len(scores))
    
    # Timestamps: nao tem temporal info, usar indice sequencial
    timestamps = np.arange(n_edges, dtype=np.int64)
    
    print(f'  Edge labels (OR): {y.sum():,} suspeitas ({100*y.mean():.1f}%)')
    print(f'  [NOTA] Sem timestamps reais. Lk baseado em vizinhanca apenas.')
    print(f'  [NOTA] Scores = node labels. Para paper, treinar GNN.')
    
    return {
        'name': name,
        'scores': scores, 'src': src, 'dst': dst,
        'timestamps': timestamps, 'y': y,
        'n_nodes': n_nodes, 'n_edges': n_edges,
        'delta_L': n_edges,  # sem temporal, Lk liga todos os vizinhos
    }


def load_aml_pretrained(base, artif_sub, probs_sub, model, seed, name):
    """Carrega AML100k/AML1M com scores de GNN pre-treinado."""
    artif = base / artif_sub
    probs = base / probs_sub
    
    npz = np.load(probs / f'{model}_seed{seed}_test.npz')
    scores = np.asarray(npz['p'], dtype=float)
    y = np.asarray(npz['y'], dtype=int)
    
    cache = torch.load(artif / 'edge_data_v4_clean.pt',
                       map_location='cpu', weights_only=False)
    ei_all = cache['ei_all_cpu']
    te_idx = cache['te_idx']
    if isinstance(te_idx, torch.Tensor): te_idx = te_idx.numpy()
    if isinstance(ei_all, torch.Tensor): ei_all = ei_all.numpy()
    src = ei_all[0, te_idx].astype(np.int64)
    dst = ei_all[1, te_idx].astype(np.int64)
    
    candidates = ['transactions.csv','transaction.csv',
                  'hi-large_trans.csv','hi-medium_trans.csv','hi-small_trans.csv',
                  'li-large_trans.csv','li-medium_trans.csv','li-small_trans.csv']
    csv_path = next((list(base.rglob(c))[0] for c in candidates
                     if list(base.rglob(c))), None)
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip().str.lower()
    time_col = next(c for c in ['timestamp','time'] if c in df.columns)
    ts_raw = pd.to_numeric(df[time_col], errors='coerce').fillna(0).astype(np.int64).values
    order = np.argsort(ts_raw, kind='stable')
    ts_sort = ts_raw[order]
    src_col = next(c for c in ['sender_account_id','src'] if c in df.columns)
    dst_col = next(c for c in ['receiver_account_id','dst'] if c in df.columns)
    mask = df[src_col].astype(str).values[order] != df[dst_col].astype(str).values[order]
    ts_clean = ts_sort[mask]
    timestamps = ts_clean[te_idx]
    
    print(f'[{name}] {len(scores):,} edges | pos={y.sum():,} ({100*y.mean():.2f}%)')
    
    return {
        'name': name,
        'scores': scores, 'src': src, 'dst': dst,
        'timestamps': timestamps, 'y': y,
        'n_nodes': int(max(src.max(), dst.max())) + 1,
        'n_edges': len(scores),
        'delta_L': 7,
    }


print('Loaders definidos: bitcoin_otc/alpha, paysim, elliptic, ibm_aml_txns, '
      'ibm_hi_small, amazon_fraud, yelp_fraud, aml_pretrained')

In [ ]:
# CELULA 5 - Carregar todos os datasets disponíveis
# Ordem: datasets menores primeiro, depois maiores
# Datasets com scores heuristicos marcados com [H]
# Datasets sem timestamps marcados com [NT]

datasets = []

# --- Grupo 1: Datasets com edge-level labels nativos ---

# AML100k/1M (pre-treinados, GNN scores)
try:
    ds = load_aml_pretrained(
        AML100K_BASE, 'artifacts', 'results/probs_v4',
        'SAGE', 42, 'AML100k')
    datasets.append(ds)
except Exception as e:
    print(f'[SKIP] AML100k: {e}')

try:
    ds = load_aml_pretrained(
        AML1M_BASE, 'artifacts',
        'results_aml1m_graphsage_only/probs_v4',
        'GraphSAGE', 44, 'AML1M')
    datasets.append(ds)
except Exception as e:
    print(f'[SKIP] AML1M: {e}')

# Bitcoin OTC / Alpha (edge ratings)
try:
    datasets.append(load_bitcoin_otc_alpha('bitcoin_otc'))
except Exception as e:
    print(f'[SKIP] Bitcoin OTC: {e}')

try:
    datasets.append(load_bitcoin_otc_alpha('bitcoin_alpha'))
except Exception as e:
    print(f'[SKIP] Bitcoin Alpha: {e}')

# IBM AML transactions.csv (1.3M, edge IS_FRAUD) [H]
try:
    datasets.append(load_ibm_aml_transactions())
except Exception as e:
    print(f'[SKIP] IBM AML txns: {e}')

# IBM HI-Small (5M, edge Is Laundering) [H]
try:
    datasets.append(load_ibm_hi_small())
except Exception as e:
    print(f'[SKIP] IBM HI-Small: {e}')

# --- Grupo 2: Datasets com node-level labels (derivar edge) ---

# Elliptic (node -> edge, has timestamps) [H]
try:
    datasets.append(load_elliptic())
except Exception as e:
    print(f'[SKIP] Elliptic: {e}')

# Amazon Fraud (.mat, node-level, no timestamps) [H][NT]
try:
    datasets.append(load_mat_fraud('amazon_fraud', 'Amazon.mat'))
except Exception as e:
    print(f'[SKIP] Amazon Fraud: {e}')

# Yelp Fraud (.mat, node-level, no timestamps) [H][NT]
try:
    datasets.append(load_mat_fraud('yelp_fraud', 'YelpChi.mat'))
except Exception as e:
    print(f'[SKIP] Yelp Fraud: {e}')

# --- Grupo 3: Datasets grandes (carregar por ultimo) ---

# PaySim (~6M txns) [H]
try:
    datasets.append(load_paysim())
except Exception as e:
    print(f'[SKIP] PaySim: {e}')

# --- Summary ---
print(f'\n{"="*70}')
print(f'  {len(datasets)} datasets carregados')
print(f'{"="*70}')
for ds in datasets:
    flags = ''
    if ds.get('delta_L', 0) == ds.get('n_edges', -1):
        flags += ' [NT]'
    print(f"  {ds['name']:20s} | {ds['n_nodes']:>10,} nodes | "
          f"{ds['n_edges']:>10,} edges | pos={ds['y'].sum():>8,} "
          f"({100*ds['y'].mean():.1f}%) | dL={ds['delta_L']}{flags}")

In [ ]:
# CELULA 6 - Rodar BTCS em todos os datasets

KS = [0.01, 0.05, 0.10]
GAMMA = 1.0
BUDGET_B = 100

all_rows = []

for ds in datasets:
    name = ds['name']
    delta_L = ds['delta_L']
    print(f'\n{"="*60}')
    print(f'  {name} ({ds["n_edges"]:,} edges, delta_L={delta_L})')
    print(f'{"="*60}')
    
    for k in KS:
        kp = f'{int(k*100)}%'
        print(f'\n  k={kp}:')
        
        with measure_resources() as res:
            cases = btcs_generic(
                ds['scores'], ds['src'], ds['dst'],
                ds['timestamps'], ds['y'],
                k=k, delta_L=delta_L,
                resolution=GAMMA, budget_B=BUDGET_B)
            m = evaluate_cases_generic(
                cases, ds['y'], ds['n_edges'], k)
        
        row = {
            'dataset': name,
            'n_nodes': ds['n_nodes'],
            'n_edges': ds['n_edges'],
            'k%': kp, 'k_frac': k,
            'delta_L': delta_L,
            'resolution': GAMMA,
            'budget_B': BUDGET_B,
            **m, **res,
        }
        all_rows.append(row)
        print(f"    cov={m['coverage']:.3f} pur={m['purity_induced']:.4f} "
              f"OCR100={m['ocr_b100']:.3f} #cases={m['n_cases']:,} "
              f"{res['time_s']:.1f}s")

df_multi = pd.DataFrame(all_rows)
print(f'\nTotal: {len(all_rows)} configuracoes em {len(datasets)} datasets.')

In [ ]:
# CELULA 7 - Tabela resumo

print('=== Resultados Multi-Dataset (BTCS v3, B=100) ===')
pivot = df_multi.pivot_table(
    index='dataset',
    columns='k%',
    values=['coverage', 'ocr_b100', 'n_cases', 'purity_induced'],
    aggfunc='first'
).round(4)
display(pivot)

# Dataset summary
print('\n=== Dataset Summary ===')
summary = df_multi.groupby('dataset').agg({
    'n_nodes': 'first', 'n_edges': 'first',
}).reset_index()
for ds in datasets:
    fraud_rate = 100 * ds['y'].mean()
    print(f"  {ds['name']:20s} | {ds['n_nodes']:>10,} nodes | "
          f"{ds['n_edges']:>10,} edges | fraud={fraud_rate:.2f}%")

In [ ]:
# CELULA 8 - Plot: Coverage por dataset

n_ds = len(datasets)
fig, axes = plt.subplots(1, min(n_ds, 3), figsize=(5*min(n_ds, 3), 5),
                         squeeze=False)
axes = axes[0]

for i, ds_info in enumerate(datasets[:3]):
    ax = axes[i]
    name = ds_info['name']
    sub = df_multi[df_multi['dataset'] == name]
    
    k_vals = sub['k_frac'].values
    covs = sub['coverage'].values
    ocrs = sub['ocr_b100'].values
    
    ax.bar(range(len(k_vals)), covs, color='#2ca02c', alpha=0.8,
           label='Coverage')
    ax2 = ax.twinx()
    ax2.plot(range(len(k_vals)), ocrs, 'r-o', ms=8, label='OCR(100)')
    
    ax.set_title(f'{name}\n({ds_info["n_edges"]:,} edges)')
    ax.set_xticks(range(len(k_vals)))
    ax.set_xticklabels([f'{int(k*100)}%' for k in k_vals])
    ax.set_xlabel('k')
    ax.set_ylabel('Coverage', color='#2ca02c')
    ax2.set_ylabel('OCR(100)', color='red')
    ax.set_ylim(0, 1.05)
    ax2.set_ylim(-0.05, 1.05)

plt.tight_layout()
plt.savefig(OUT / 'multi_dataset_coverage.png', dpi=150, bbox_inches='tight')
plt.show()

# Se mais de 3 datasets, segundo plot
if n_ds > 3:
    fig, axes = plt.subplots(1, n_ds - 3, figsize=(5*(n_ds-3), 5),
                             squeeze=False)
    axes = axes[0]
    for i, ds_info in enumerate(datasets[3:]):
        ax = axes[i]
        name = ds_info['name']
        sub = df_multi[df_multi['dataset'] == name]
        k_vals = sub['k_frac'].values
        covs = sub['coverage'].values
        ax.bar(range(len(k_vals)), covs, color='#2ca02c', alpha=0.8)
        ax.set_title(f'{name}\n({ds_info["n_edges"]:,} edges)')
        ax.set_xticks(range(len(k_vals)))
        ax.set_xticklabels([f'{int(k*100)}%' for k in k_vals])
        ax.set_ylabel('Coverage')
        ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.savefig(OUT / 'multi_dataset_coverage_2.png', dpi=150, bbox_inches='tight')
    plt.show()

print('Plots salvos.')

In [ ]:
# CELULA 9 - Export

df_multi.to_csv(OUT / 'multi_dataset_results.csv', index=False)
print(f'CSV: {OUT}/multi_dataset_results.csv ({len(df_multi)} rows)')

# LaTeX
cols_tex = ['dataset', 'k%', 'n_cases', 'coverage', 'purity_induced',
            'ocr_b100', 'edges_per_case_median', 'time_s']
df_multi[cols_tex].round(4).to_latex(
    OUT / 'multi_dataset_table.tex', index=False, escape=False)
print(f'LaTeX: {OUT}/multi_dataset_table.tex')

print('\n=== Progresso do Paper ===')
print('1.[ok] nb01 - Baselines (B0-B3)')
print('2.[ok] nb02 - BTCS v3 (WCC+Leiden)')
print('3.[ok] nb03 - Ablations (delta_L, B, arch)')
print('4.[ok] nb04 - Multi-Dataset Evaluation (10 datasets)')
print('5.[ok] NP-hardness proof (docs/)')
print('\nDatasets com scores heuristicos (treinar GNN):')
print('  - IBM AML txns, IBM HI-Small, PaySim, Elliptic')
print('  - Amazon Fraud, Yelp Fraud (tambem sem timestamps)')
print('\nProximos passos:')
print('  - Treinar GNNs para datasets com scores heuristicos')
print('  - Implementar baselines (METIS, Louvain+constraint)')
print('  - Draft do paper ICDM 2026')